# Fourier Transform — Engineering Depth
**Not the physics surface-level version**

Physics courses: $\tilde{f}(k) = \int f(x)e^{-ikx}dx$. Done.

Engineering reality: conventions, DFT bins, FFT algorithm, spectral leakage,
windowing, zero-padding, 2D FT, transfer functions, Z-transform connection,
and exactly how all of it connects to the dispersive GS receiver.

| § | Topic |
|---|-------|
| 1 | Four Fourier transforms — conventions and when to use each |
| 2 | DFT: bins, resolution, aliasing |
| 3 | FFT algorithm — why O(N log N) |
| 4 | Spectral leakage and windowing |
| 5 | Convolution theorem — the engineering workhorse |
| 6 | Transfer functions H(ω) — LTI systems |
| 7 | Z-transform connection |
| 8 | 2D FFT — images, diffraction, optical systems |
| 9 | Short-time FT (STFT) and spectrograms |
| 10 | FT in the dispersive GS receiver — exact derivation |

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import windows, spectrogram, freqz
from scipy.fft import fft, ifft, fftfreq, fftshift, rfft, rfftfreq
import warnings; warnings.filterwarnings('ignore')
plt.rcParams.update({'figure.dpi': 130, 'font.size': 11})

---
## §1 · Four Fourier transforms — conventions

| Transform | Domain | Formula | Used in |
|-----------|--------|---------|--------|
| **CTFT** | Continuous, aperiodic | $X(f)=\int_{-\infty}^{\infty}x(t)e^{-j2\pi ft}dt$ | Signals, circuits |
| **CTFS** | Continuous, periodic | $c_n=\frac{1}{T}\int_0^T x(t)e^{-jn\omega_0 t}dt$ | Power spectra |
| **DTFT** | Discrete, aperiodic | $X(e^{j\omega})=\sum_{n=-\infty}^{\infty}x[n]e^{-j\omega n}$ | DSP theory |
| **DFT** | Discrete, periodic | $X[k]=\sum_{n=0}^{N-1}x[n]e^{-j2\pi kn/N}$ | Computation |

**The one confusing thing:** physicists write $e^{-ikx}$, engineers write $e^{-j2\pi ft}$. Same transform, different variable.

**NumPy convention:** `np.fft.fft` uses $X[k]=\sum_{n}x[n]e^{-j2\pi kn/N}$ — the engineering DFT.

In [ ]:
# Demonstrate all four on the same signal: a pure 50 Hz tone
fs   = 1000.0      # sampling rate Hz
T    = 1.0         # duration s
N    = int(fs*T)   # samples
t    = np.arange(N) / fs
f0   = 50.0        # signal frequency Hz
x    = np.sin(2*np.pi*f0*t) + 0.5*np.cos(2*np.pi*3*f0*t)

# DFT via NumPy FFT
X     = fft(x)
freqs = fftfreq(N, 1/fs)   # Hz

fig, axes = plt.subplots(2, 2, figsize=(13, 7))

# Time domain
axes[0,0].plot(t[:200]*1e3, x[:200], lw=1.5, color='steelblue')
axes[0,0].set_xlabel('Time (ms)'); axes[0,0].set_ylabel('Amplitude')
axes[0,0].set_title('x(t) — time domain')
axes[0,0].grid(True, alpha=0.3)

# Magnitude spectrum (one-sided)
f_pos = freqs[:N//2]
X_pos = np.abs(X[:N//2]) * 2/N
axes[0,1].stem(f_pos[:200], X_pos[:200], markerfmt='C1o', linefmt='C1-', basefmt='k-')
axes[0,1].set_xlabel('Frequency (Hz)'); axes[0,1].set_ylabel('|X(f)|')
axes[0,1].set_title('DFT magnitude spectrum')
axes[0,1].grid(True, alpha=0.3)

# Phase spectrum
X_phase = np.angle(X[:N//2])
mask = X_pos > 0.05  # only show non-noise phases
axes[1,0].scatter(f_pos[mask], np.degrees(X_phase[mask]),
                  color='crimson', s=40, zorder=5)
axes[1,0].set_xlabel('Frequency (Hz)'); axes[1,0].set_ylabel('Phase (°)')
axes[1,0].set_title('DFT phase spectrum (signal bins only)')
axes[1,0].set_xlim(0, 250); axes[1,0].grid(True, alpha=0.3)

# Two-sided spectrum (fftshift)
axes[1,1].plot(fftshift(freqs), np.abs(fftshift(X))/N, lw=1.5, color='#2ca02c')
axes[1,1].set_xlabel('Frequency (Hz)'); axes[1,1].set_ylabel('|X[k]|/N')
axes[1,1].set_title('Two-sided spectrum (fftshift)')
axes[1,1].grid(True, alpha=0.3)

plt.suptitle('DFT of $x(t) = \\sin(2\\pi\\cdot50t) + 0.5\\cos(2\\pi\\cdot150t)$',
             fontweight='bold')
plt.tight_layout(); plt.show()

print('DFT conventions summary:')
print(f'  N={N} samples,  fs={fs} Hz,  T={T} s')
print(f'  Frequency resolution: Δf = fs/N = {fs/N:.3f} Hz')
print(f'  Nyquist frequency:    f_max = fs/2 = {fs/2:.1f} Hz')
print(f'  Bin k maps to frequency: f = k·(fs/N)')

## §2 · DFT internals — bins, resolution, aliasing

**Frequency resolution:** $\Delta f = f_s/N = 1/T$ — more samples OR longer record → finer resolution.

**Bin mapping:** bin $k$ ↔ frequency $f_k = k \cdot f_s/N$ (for $k < N/2$).

**Nyquist–Shannon:** sample at $f_s$ → can represent frequencies up to $f_s/2$ without aliasing.

**Aliasing:** $f_{alias} = |f_{signal} - n \cdot f_s|$ for integer $n$ — signal at 750 Hz sampled at 1 kHz aliases to 250 Hz.

In [ ]:
# Aliasing demonstration
fs_low = 200.0   # undersample
f_sig  = 150.0   # signal frequency
t_alias = np.arange(0, 1.0, 1/fs_low)
t_true  = np.arange(0, 1.0, 1/2000)

x_sampled = np.sin(2*np.pi*f_sig*t_alias)
x_true    = np.sin(2*np.pi*f_sig*t_true)
f_alias   = abs(f_sig - fs_low)   # 150-200 = -50 → |50|
x_alias   = np.sin(2*np.pi*f_alias*t_true)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].plot(t_true, x_true,   lw=1, color='steelblue', alpha=0.6, label=f'{f_sig:.0f} Hz (true)')
axes[0].plot(t_true, x_alias,  lw=1.5, color='crimson',  ls='--',   label=f'{f_alias:.0f} Hz (alias)')
axes[0].stem(t_alias, x_sampled, markerfmt='ko', linefmt='k-', basefmt='k-',
             label=f'Samples @ {fs_low:.0f} Hz')
axes[0].set_xlabel('Time (s)'); axes[0].set_title(f'Aliasing: {f_sig:.0f} Hz at fs={fs_low:.0f} Hz → aliases to {f_alias:.0f} Hz')
axes[0].legend(fontsize=8); axes[0].set_xlim(0, 0.1)

# Zero-padding: more DFT points → interpolated spectrum (not more resolution)
N_orig = 64
t_short = np.arange(N_orig) / fs
x_short = np.sin(2*np.pi*f0*t_short)

for N_pad, col, lbl in [(64,'steelblue','No padding'), (256,'crimson','4× zero-pad'), (1024,'#2ca02c','16× zero-pad')]:
    X_pad = fft(x_short, n=N_pad)
    f_pad = rfftfreq(N_pad, 1/fs)
    axes[1].plot(f_pad[:N_pad//4], np.abs(rfft(x_short, n=N_pad))[:N_pad//4]/N_orig,
                 lw=1.5, color=col, label=lbl)

axes[1].set_xlabel('Frequency (Hz)'); axes[1].set_ylabel('|X|/N')
axes[1].set_title('Zero-padding: interpolates spectrum, does NOT improve resolution')
axes[1].legend(fontsize=9); axes[1].set_xlim(0, 150)
axes[1].grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

## §3 · FFT algorithm — why O(N log N)

**Naive DFT:** $X[k]=\sum_{n=0}^{N-1}x[n]e^{-j2\pi kn/N}$ — $N$ outputs × $N$ multiplications = $O(N^2)$.

**Cooley–Tukey FFT (1965):** split even/odd samples recursively.
$$X[k] = \underbrace{\sum_{n\text{ even}} x[n]e^{-j2\pi kn/N}}_{E[k]} + e^{-j2\pi k/N}\underbrace{\sum_{n\text{ odd}} x[n]e^{-j2\pi kn/N}}_{O[k]}$$

Each half is an $N/2$-point DFT → $T(N) = 2T(N/2) + O(N)$ → $O(N\log_2 N)$.

In [ ]:
# Verify FFT vs. naive DFT, then benchmark complexity

def naive_dft(x):
    N = len(x)
    n = np.arange(N)
    k = n[:, None]          # column
    W = np.exp(-2j*np.pi*k*n/N)   # twiddle factor matrix
    return W @ x

# Verify equality
x_test = np.random.randn(64) + 1j*np.random.randn(64)
X_naive = naive_dft(x_test)
X_fft   = fft(x_test)
print(f'Max error (naive DFT vs FFT): {np.max(np.abs(X_naive - X_fft)):.2e}')

# Benchmark
import time
sizes = [2**k for k in range(4, 15)]
t_fft, t_naive = [], []

for N_b in sizes:
    x_b = np.random.randn(N_b)
    t0 = time.perf_counter(); fft(x_b); t_fft.append(time.perf_counter()-t0)
    if N_b <= 2048:
        t0 = time.perf_counter(); naive_dft(x_b); t_naive.append(time.perf_counter()-t0)
    else:
        t_naive.append(np.nan)

fig, ax = plt.subplots(figsize=(9, 5))
ax.loglog(sizes, t_fft,   'o-', color='steelblue', lw=2, label='FFT  O(N log N)')
ax.loglog(sizes[:7], [t for t in t_naive if not np.isnan(t)], 's--',
          color='crimson', lw=2, label='Naive DFT  O(N²)')
# Reference lines
N_ref = np.array(sizes, dtype=float)
ax.loglog(N_ref, N_ref*np.log2(N_ref)*t_fft[-1]/(N_ref[-1]*np.log2(N_ref[-1])),
          ':', color='steelblue', alpha=0.5, label='N log N reference')
ax.loglog(N_ref[:7], N_ref[:7]**2 * t_naive[4]/(sizes[4]**2),
          ':', color='crimson', alpha=0.5, label='N² reference')
ax.set_xlabel('N (samples)'); ax.set_ylabel('Time (s)')
ax.set_title('FFT vs. naive DFT timing')
ax.legend(); ax.grid(True, which='both', alpha=0.3)
plt.tight_layout(); plt.show()

## §4 · Spectral leakage and windowing

**Leakage cause:** DFT assumes the $N$-sample record is one period of a periodic signal.
If the signal frequency doesn't fall exactly on a bin, the discontinuity at the record edges
spreads energy across all bins — **spectral leakage**.

**Fix:** multiply by a **window** $w[n]$ that tapers to zero at the edges.

In [ ]:
N_w  = 512
fs_w = 1000.0
# Non-integer number of cycles → leakage
f_leak = 50.7   # not on a bin
t_w = np.arange(N_w) / fs_w
x_w = np.sin(2*np.pi*f_leak*t_w)

window_funcs = {
    'Rectangular': np.ones(N_w),
    'Hann':        windows.hann(N_w),
    'Hamming':     windows.hamming(N_w),
    'Blackman':    windows.blackman(N_w),
    'Flat-top':    windows.flattop(N_w),
}

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
f_ax = rfftfreq(N_w, 1/fs_w)
colors_w = ['#1f77b4','#ff7f0e','#2ca02c','#d62728','#9467bd']

for (name, w), col in zip(window_funcs.items(), colors_w):
    X_w = rfft(x_w * w)
    X_dB = 20*np.log10(np.abs(X_w)/np.max(np.abs(X_w)) + 1e-15)
    axes[0].plot(f_ax, X_dB, lw=1.5, color=col, label=name)
    axes[1].plot(t_w[:N_w//4]*1e3, w[:N_w//4], lw=1.5, color=col, label=name)

axes[0].set_xlim(0, 200); axes[0].set_ylim(-100, 5)
axes[0].set_xlabel('Frequency (Hz)'); axes[0].set_ylabel('Magnitude (dB)')
axes[0].set_title(f'Spectral leakage for f={f_leak} Hz — window comparison')
axes[0].axvline(f_leak, ls='--', color='gray', alpha=0.7, label=f'{f_leak} Hz')
axes[0].legend(fontsize=8, ncol=2)
axes[0].grid(True, alpha=0.3)

axes[1].set_xlabel('Time (ms)'); axes[1].set_ylabel('w[n]')
axes[1].set_title('Window functions (first quarter)')
axes[1].legend(fontsize=8); axes[1].grid(True, alpha=0.3)

plt.suptitle('Windowing reduces spectral leakage at cost of main-lobe width',
             fontweight='bold')
plt.tight_layout(); plt.show()

# Window properties table
print(f'{"Window":<12}  {"Main lobe (bins)":<18}  {"Peak sidelobe (dB)":<20}  {"Best for"}')
print('-'*72)
props = [
    ('Rect',     '1',  '-13',  'Coherent signals on-bin'),
    ('Hann',     '2',  '-32',  'General purpose (most common)'),
    ('Hamming',  '2',  '-43',  'Narrowband signals'),
    ('Blackman', '3',  '-58',  'Dynamic range > 60 dB'),
    ('Flat-top', '5',  '-93',  'Amplitude measurement accuracy'),
]
for row in props:
    print(f'{row[0]:<12}  {row[1]:<18}  {row[2]:<20}  {row[3]}')

## §5 · Convolution theorem

$$\mathcal{F}\{f * g\} = \mathcal{F}\{f\} \cdot \mathcal{F}\{g\}$$

**Why it matters:** convolution in time is pointwise multiplication in frequency — and FFT makes this $O(N\log N)$ instead of $O(N^2)$.

**Every LTI system** is characterized by its impulse response $h[n]$. Output = input $*$ $h$.
In frequency: $Y(f) = X(f) \cdot H(f)$.

In [ ]:
# FFT convolution vs. direct — speed and correctness
N_conv = 1024
x_c = np.random.randn(N_conv)
h_c = np.exp(-np.arange(N_conv//4) / 50.0)  # exponential decay impulse response

# Direct convolution (O(N²))
y_direct = np.convolve(x_c, h_c, mode='full')[:N_conv]

# FFT convolution (O(N log N))
N_fft_conv = N_conv + len(h_c) - 1
X_c = fft(x_c, n=N_fft_conv)
H_c = fft(h_c, n=N_fft_conv)
y_fft_conv = np.real(ifft(X_c * H_c))[:N_conv]

print(f'Max error (direct vs FFT convolution): {np.max(np.abs(y_direct - y_fft_conv)):.2e}')

# Visualize: filtering a noisy signal
t_c = np.arange(N_conv) / fs_w
signal_clean = np.sin(2*np.pi*50*t_c)
noise_signal  = signal_clean + 0.8*np.random.randn(N_conv)

# Low-pass filter via frequency-domain multiplication
X_noisy = fft(noise_signal)
freqs_c = fftfreq(N_conv, 1/fs_w)
H_lp    = (np.abs(freqs_c) < 80).astype(float)   # brick-wall LPF at 80 Hz
y_filt  = np.real(ifft(X_noisy * H_lp))

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].plot(t_c[:300]*1e3, noise_signal[:300], alpha=0.5, lw=1, label='Noisy input', color='gray')
axes[0].plot(t_c[:300]*1e3, y_filt[:300], lw=2, color='steelblue', label='LPF output')
axes[0].plot(t_c[:300]*1e3, signal_clean[:300], '--', lw=1.5, color='crimson', label='True signal')
axes[0].set_xlabel('Time (ms)'); axes[0].set_title('Frequency-domain low-pass filtering')
axes[0].legend(fontsize=9)

axes[1].plot(fftshift(freqs_c), np.abs(fftshift(X_noisy))/N_conv, lw=1, color='gray', alpha=0.7, label='Input')
axes[1].plot(fftshift(freqs_c), np.abs(fftshift(fft(y_filt)))/N_conv, lw=2, color='steelblue', label='Filtered')
axes[1].axvline(80, ls='--', color='crimson', label='Cutoff 80 Hz')
axes[1].axvline(-80, ls='--', color='crimson')
axes[1].set_xlabel('Frequency (Hz)'); axes[1].set_xlim(-200, 200)
axes[1].set_title('Convolution theorem: Y(f) = X(f)·H(f)')
axes[1].legend(fontsize=9)
plt.tight_layout(); plt.show()

## §6 · Transfer functions H(ω) — LTI systems

For any LTI system described by differential equation:
$$a_n y^{(n)} + \cdots + a_0 y = b_m x^{(m)} + \cdots + b_0 x$$

Take FT → $H(\omega) = Y(\omega)/X(\omega) = \frac{b_m(j\omega)^m + \cdots + b_0}{a_n(j\omega)^n + \cdots + a_0}$

**Dispersive fiber transfer function** (this project):
$$H(\nu) = e^{j\pi D\nu^2}$$
This is a pure **phase filter** — $|H|=1$ everywhere, only phase changes.

In [ ]:
# RC low-pass filter transfer function
omega = np.logspace(0, 6, 1000)  # rad/s
RC_vals = [1e-3, 1e-4, 1e-5]    # RC time constants

fig, axes = plt.subplots(2, 2, figsize=(13, 8))

for RC, col in zip(RC_vals, ['steelblue','crimson','#2ca02c']):
    H = 1 / (1 + 1j*omega*RC)
    omega_c = 1/RC
    axes[0,0].semilogx(omega, 20*np.log10(np.abs(H)), lw=2, color=col,
                       label=f'RC={RC:.0e}s  ωc={omega_c:.0e}')
    axes[0,1].semilogx(omega, np.degrees(np.angle(H)), lw=2, color=col)

axes[0,0].axhline(-3, ls='--', color='gray', alpha=0.7, label='-3 dB')
axes[0,0].set_xlabel('ω (rad/s)'); axes[0,0].set_ylabel('|H(ω)| (dB)')
axes[0,0].set_title('RC LPF — Bode magnitude')
axes[0,0].legend(fontsize=8); axes[0,0].grid(True, which='both', alpha=0.3)
axes[0,1].set_xlabel('ω (rad/s)'); axes[0,1].set_ylabel('∠H(ω) (°)')
axes[0,1].set_title('RC LPF — Bode phase')
axes[0,1].grid(True, which='both', alpha=0.3)

# Dispersive fiber transfer function (this project)
nu = np.linspace(-0.5, 0.5, 1000)
for D_val, col, lbl in [(-5000,'steelblue','D=-5000'), (-5750,'crimson','D=-5750')]:
    H_disp = np.exp(1j * np.pi * D_val * nu**2)
    axes[1,0].plot(nu, np.abs(H_disp), lw=2, color=col, label=lbl)
    axes[1,1].plot(nu, np.unwrap(np.angle(H_disp)), lw=2, color=col, label=lbl)

axes[1,0].set_xlabel('Normalized frequency ν'); axes[1,0].set_ylabel('|H(ν)|')
axes[1,0].set_title('Dispersive fiber: $|H(\\nu)| = 1$ (pure phase filter)')
axes[1,0].legend()
axes[1,1].set_xlabel('Normalized frequency ν'); axes[1,1].set_ylabel('∠H(ν) (rad)')
axes[1,1].set_title('Dispersive fiber: $\\angle H(\\nu) = \\pi D \\nu^2$ (quadratic phase)')
axes[1,1].legend()

plt.suptitle('Transfer functions: RC filter vs. dispersive fiber', fontweight='bold')
plt.tight_layout(); plt.show()

print('Key insight: dispersive fiber is an ALL-PASS filter.')
print('It does NOT attenuate any frequency — only scrambles the phase quadratically.')
print('This is why GS can recover the phase: the information is not lost, just rearranged.')

## §7 · Z-transform — DT system design

$$X(z) = \sum_{n=-\infty}^{\infty} x[n]\,z^{-n}, \qquad z = e^{j\omega}\text{ on unit circle}$$

**DTFT ↔ Z-transform:** evaluate $X(z)$ on the unit circle $|z|=1$.

**Poles and zeros** determine frequency response — poles near unit circle → resonance.

In [ ]:
from scipy.signal import zpk2tf, freqz

# Design a 2nd-order IIR notch filter at 60 Hz (power-line interference)
fs_z  = 1000.0
f_notch = 60.0
omega_n = 2*np.pi*f_notch/fs_z   # normalized
r_zero  = 1.0    # zeros ON unit circle (perfect null)
r_pole  = 0.95   # poles slightly inside (narrow notch)

zeros = np.array([np.exp(1j*omega_n), np.exp(-1j*omega_n)])
poles = r_pole * zeros

b, a = zpk2tf(zeros, poles, 1.0)
w, H_notch = freqz(b, a, fs=fs_z, worN=2048)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Pole-zero plot
theta_unit = np.linspace(0, 2*np.pi, 200)
axes[0].plot(np.cos(theta_unit), np.sin(theta_unit), 'k--', lw=0.8, alpha=0.5)
axes[0].scatter(np.real(zeros), np.imag(zeros), s=80, marker='o',
                facecolors='none', edgecolors='steelblue', lw=2, label='Zeros')
axes[0].scatter(np.real(poles), np.imag(poles), s=80, marker='x',
                color='crimson', lw=2, label='Poles')
axes[0].axhline(0,'k',lw=0.5,alpha=0.3); axes[0].axvline(0,'k',lw=0.5,alpha=0.3)
axes[0].set_aspect('equal'); axes[0].set_xlabel('Re'); axes[0].set_ylabel('Im')
axes[0].set_title('Pole-zero plot (z-plane)'); axes[0].legend(fontsize=8)

# Bode magnitude
axes[1].plot(w, 20*np.log10(np.abs(H_notch)+1e-15), lw=2, color='steelblue')
axes[1].axvline(f_notch, ls='--', color='gray', label=f'{f_notch} Hz notch')
axes[1].set_xlabel('Frequency (Hz)'); axes[1].set_ylabel('|H| (dB)')
axes[1].set_title('60 Hz notch filter — magnitude response')
axes[1].legend(); axes[1].grid(True, alpha=0.3)

# Demonstrate on contaminated signal
t_z   = np.arange(2000)/fs_z
x_ecg = np.sin(2*np.pi*1.2*t_z) + 0.5*np.sin(2*np.pi*3*t_z)  # mock ECG harmonics
x_noise = x_ecg + 0.8*np.sin(2*np.pi*f_notch*t_z)             # + 60 Hz hum
from scipy.signal import lfilter
y_clean = lfilter(b, a, x_noise)
axes[2].plot(t_z[:400]*1e3, x_noise[:400], alpha=0.5, lw=1, color='gray', label='Contaminated')
axes[2].plot(t_z[:400]*1e3, y_clean[:400], lw=2, color='steelblue', label='Notch filtered')
axes[2].set_xlabel('Time (ms)'); axes[2].set_title('60 Hz hum removal')
axes[2].legend(fontsize=9)

plt.tight_layout(); plt.show()

## §8 · 2D FFT — images, diffraction, optical systems

$$\mathcal{F}\{f(x,y)\}(u,v) = \int\int f(x,y)\,e^{-j2\pi(ux+vy)}\,dx\,dy$$

**Fraunhofer diffraction:** the far-field diffraction pattern of an aperture $A(x,y)$ is exactly $|\mathcal{F}\{A\}|^2$ — the 2D FT power spectrum.

**Optical connection to this project:** the dispersive Fourier transform in fiber is the 1D analog of free-space Fraunhofer diffraction.

In [ ]:
from scipy.fft import fft2, fftshift as fftshift2

N2 = 256
x2 = np.linspace(-1, 1, N2)
X2, Y2 = np.meshgrid(x2, x2)

apertures = {
    'Circular':   (X2**2 + Y2**2) < 0.15**2,
    'Rectangular': (np.abs(X2) < 0.15) & (np.abs(Y2) < 0.10),
    'Double slit': ((np.abs(X2 - 0.12) < 0.02) | (np.abs(X2 + 0.12) < 0.02)) & (np.abs(Y2) < 0.15),
    'Annular':     ((X2**2+Y2**2) < 0.20**2) & ((X2**2+Y2**2) > 0.10**2),
}

fig, axes = plt.subplots(2, 4, figsize=(16, 7))
for col_idx, (name, aperture) in enumerate(apertures.items()):
    A = aperture.astype(complex)
    FT = fftshift2(fft2(A))
    diffraction = np.abs(FT)**2

    axes[0, col_idx].imshow(aperture, cmap='gray', aspect='equal')
    axes[0, col_idx].set_title(f'{name}\naperture')
    axes[0, col_idx].axis('off')

    # Log scale for diffraction pattern
    axes[1, col_idx].imshow(np.log1p(diffraction), cmap='inferno', aspect='equal')
    axes[1, col_idx].set_title('Diffraction\npattern (log)')
    axes[1, col_idx].axis('off')

plt.suptitle('2D FFT = Fraunhofer diffraction: aperture → far-field intensity pattern',
             fontweight='bold', fontsize=12)
plt.tight_layout(); plt.show()

print('Double-slit interference fringes appear as expected — Young\'s experiment via FFT.')

## §9 · Short-time FT (STFT) and spectrograms

$$\text{STFT}\{x\}(t,f) = \int x(\tau)\,w(\tau-t)\,e^{-j2\pi f\tau}\,d\tau$$

**Trade-off:** narrow window → good time resolution, poor frequency resolution. Wide window → reverse.

**Uncertainty principle for signals:** $\Delta t \cdot \Delta f \geq 1/(4\pi)$ (Gabor limit).

**Connection to this project:** the chirped signal after dispersion is a frequency-swept waveform — its instantaneous frequency is $f(t) \propto Dt$, visible on a spectrogram.

In [ ]:
import sys; sys.path.insert(0, '..')
try:
    from gs_core import disperse, make_measurements
    HAS_GSCORE = True
except ImportError:
    HAS_GSCORE = False

fs_s = 1000.0
t_s  = np.linspace(0, 1, 1000)

# Chirp signal (linear FM sweep)
f_start, f_end = 10.0, 200.0
chirp_sig = np.sin(2*np.pi*(f_start*t_s + (f_end-f_start)/2*t_s**2))

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# Time domain
axes[0].plot(t_s, chirp_sig, lw=1, color='steelblue')
axes[0].set_xlabel('Time (s)'); axes[0].set_title('Chirp signal x(t)')

# STFT spectrogram
f_spec, t_spec, Sxx = spectrogram(chirp_sig, fs=fs_s, window=windows.hann(64),
                                   nperseg=64, noverlap=60, scaling='spectrum')
axes[1].pcolormesh(t_spec, f_spec, 10*np.log10(Sxx+1e-15), cmap='inferno',
                   shading='auto')
axes[1].set_xlabel('Time (s)'); axes[1].set_ylabel('Frequency (Hz)')
axes[1].set_title('STFT spectrogram (64-sample Hann window)')

# After dispersion: QPSK signal becomes frequency-swept
if HAS_GSCORE:
    d  = make_measurements('QPSK', n_symbols=64, sps=8, D1=-5000., D2=-5750., snr_db=30.)
    I1 = d['I1'][:512]
    f_d, t_d, Sxx_d = spectrogram(I1, fs=1.0, window=windows.hann(32),
                                   nperseg=32, noverlap=28, scaling='spectrum')
    axes[2].pcolormesh(t_d, f_d, 10*np.log10(Sxx_d+1e-15), cmap='inferno', shading='auto')
    axes[2].set_xlabel('Sample'); axes[2].set_ylabel('Normalized freq')
    axes[2].set_title('Dispersed QPSK intensity I₁(t) — spectrogram')
else:
    axes[2].text(0.5, 0.5, 'gs_core not found\n(run from project root)',
                 ha='center', va='center', transform=axes[2].transAxes)

plt.suptitle('STFT: time-frequency analysis — chirp and dispersed optical signal',
             fontweight='bold')
plt.tight_layout(); plt.show()

## §10 · FT in the dispersive GS receiver — exact derivation

This is where all of §1–§9 connects directly to the project.

**Starting point:** optical field $E(t)$ with unknown phase $\phi(t)$.

**Step 1 — FT to frequency domain:**
$$\hat{E}(\nu) = \mathcal{F}\{E(t)\} = \int E(t)e^{-j2\pi\nu t}dt$$

**Step 2 — Apply dispersion (transfer function in frequency domain):**
$$\hat{E}_d(\nu) = \hat{E}(\nu) \cdot H(\nu), \quad H(\nu) = e^{j\pi D\nu^2}$$

**Step 3 — IFT back to time domain:**
$$E_d(t) = \mathcal{F}^{-1}\{\hat{E}_d(\nu)\} = \mathcal{F}^{-1}\{\hat{E}(\nu)\cdot e^{j\pi D\nu^2}\}$$

**Step 4 — Photodetect:** $I_d(t) = |E_d(t)|^2$ — phase is lost, intensity is measured.

**GS recovers phase by:** alternating constraint projections between two dispersed domains $D_1$, $D_2$, using the FT/IFT as the core operation.

In [ ]:
if HAS_GSCORE:
    from gs_core import disperse, retrieve_phase, make_measurements

    # Trace every FT step for one signal
    d = make_measurements('QPSK', n_symbols=64, sps=8, D1=-5000., D2=-5750., snr_db=40.)
    E      = d['E']       # true complex field
    N_tr   = len(E)
    nu     = np.fft.fftfreq(N_tr)   # normalized frequency axis

    # Step-by-step
    E_hat  = np.fft.fft(E)                                    # step 1: FT
    H1     = np.exp(1j*np.pi*d['D1']*nu**2)                  # step 2a: H(ν)
    Ed_hat = E_hat * H1                                       # step 2b: multiply
    E_d    = np.fft.ifft(Ed_hat)                              # step 3: IFT
    I1_calc = np.abs(E_d)**2                                  # step 4: detect

    print(f'Max error |I1_calc - I1_gs_core|: {np.max(np.abs(I1_calc - d["I1"])):.2e}')

    fig, axes = plt.subplots(2, 3, figsize=(16, 8))

    axes[0,0].plot(np.abs(E[:200]), lw=1.5, color='steelblue')
    axes[0,0].set_title('|E(t)| — amplitude'); axes[0,0].set_xlabel('Sample')

    axes[0,1].plot(nu, np.abs(np.fft.fftshift(E_hat)), lw=1.5, color='crimson')
    axes[0,1].set_title('|Ê(ν)| — FT magnitude'); axes[0,1].set_xlabel('Normalized ν')

    axes[0,2].plot(nu, np.angle(np.fft.fftshift(H1)), lw=1.5, color='#2ca02c')
    axes[0,2].set_title(f'∠H(ν) = πDν²  (D={d["D1"]:.0f})')
    axes[0,2].set_xlabel('Normalized ν')

    axes[1,0].plot(np.abs(E_d[:200]), lw=1.5, color='#984ea3')
    axes[1,0].set_title('|E_d(t)| — dispersed amplitude'); axes[1,0].set_xlabel('Sample')

    axes[1,1].plot(I1_calc[:200], lw=1.5, color='steelblue', label='Step-by-step')
    axes[1,1].plot(d['I1'][:200], '--', lw=1.5, color='crimson', label='gs_core.disperse')
    axes[1,1].set_title('I₁(t) = |E_d(t)|² — measured intensity')
    axes[1,1].legend(fontsize=9); axes[1,1].set_xlabel('Sample')

    phi_rec, errs = retrieve_phase(d['I1'], d['I2'], d['D1'], d['D2'], n_iter=50)
    off = np.angle(np.mean(np.exp(1j*(d['phi_true']-phi_rec))))
    axes[1,2].plot(d['phi_true'][:200], lw=1.5, color='k', label='True φ(t)')
    axes[1,2].plot((phi_rec+off)[:200], '--', lw=1.5, color='steelblue', label='GS recovered')
    rms = np.degrees(np.sqrt(np.mean(np.angle(np.exp(1j*(phi_rec+off-d['phi_true'])))**2)))
    axes[1,2].set_title(f'GS phase recovery  RMS={rms:.1f}°')
    axes[1,2].legend(fontsize=9); axes[1,2].set_xlabel('Sample')

    plt.suptitle('Every FT step in the dispersive GS receiver', fontweight='bold')
    plt.tight_layout(); plt.show()
else:
    print('Run this notebook from the project root directory to access gs_core.')

---
## Engineering FT cheat sheet

| Property | Formula | Engineering use |
|----------|---------|----------------|
| Linearity | $\mathcal{F}\{af+bg\}=a\hat{f}+b\hat{g}$ | Superposition |
| Time shift | $\mathcal{F}\{f(t-t_0)\}=e^{-j2\pi f t_0}\hat{f}$ | Delay → phase ramp |
| Freq shift | $\mathcal{F}\{e^{j2\pi f_0 t}f(t)\}=\hat{f}(f-f_0)$ | Modulation |
| Convolution | $\mathcal{F}\{f*g\}=\hat{f}\cdot\hat{g}$ | LTI filtering |
| Correlation | $\mathcal{F}\{f\star g\}=\hat{f}^*\cdot\hat{g}$ | Matched filter |
| Parseval | $\int|f|^2dt=\int|\hat{f}|^2df$ | Energy conservation |
| Derivative | $\mathcal{F}\{f'\}=j2\pi f\hat{f}$ | Diff eq → algebraic |
| Scaling | $\mathcal{F}\{f(at)\}=\frac{1}{|a|}\hat{f}(f/a)$ | Bandwidth tradeoff |
| Disp. fiber | $\hat{E}_d=\hat{E}\cdot e^{j\pi D\nu^2}$ | This project |

**NumPy pitfalls:**
- `np.fft.fftfreq(N)` returns $[-0.5, 0.5)$ normalized — multiply by $f_s$ for Hz
- `np.fft.fftshift` moves zero-frequency to center — use for display only
- `np.fft.rfft` for real signals — only positive frequencies, $N//2+1$ bins
- Normalization: `fft(x)/N` gives amplitude-correct spectrum